In [0]:
%run ./adls_auth

In [0]:
%run ./control_table

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("year_month", "")
dbutils.widgets.text("status", "SUCCESS") 

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
year_month = dbutils.widgets.get("year_month").strip()
status = dbutils.widgets.get("status").strip()

started_at = datetime.utcnow()

try:
    if not year_month:
        raise ValueError("Parameter 'year_month' was not provided.")

    year = year_month[:4]
    month = year_month[5:7]
    file_path = f"abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips_raw/year={year}/month={month}"
    
    # If the run was skipped by ADF, rows written in THIS pipeline run is 0
    if status in ("SKIPPED_ALREADY_EXISTS", "FAILED"):
        rows_written = 0
    else:  # status == "SUCCESS"
        try:
            trips_df = spark.read.parquet(file_path)
            rows_written = trips_df.count()
            if rows_written == 0:
                print(f"WARNING: status=SUCCESS but 0 rows found at {file_path} — possible silent data issue.")
        except Exception as read_err:
            print(f"ERROR: status=SUCCESS but failed to read {file_path}: {read_err}")
            status = "SUCCESS_BUT_UNREADABLE"
            rows_written = 0

    log_ingestion_event(
        spark,
        batch_id=pipeline_run_id or str(uuid.uuid4()),
        source_name="tlc_yellow_trips_adf",
        partition_key=year_month,
        status=status,
        rows_written=rows_written,
        started_at=started_at,
    )
    print(f"Logged partition '{year_month}' with status '{status}' ({rows_written:,} rows written)")

except Exception as e:
    print(f"Failed to log ADF partition summary for '{year_month}': {e}")
    raise

In [0]:
# # quick sanity check 
# log_df = spark.read.format("delta").load("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/ingestion_log")
# display(log_df.orderBy("partition_key"))

# trips_df = spark.read.parquet("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips_raw")
# print(f"Total rows landed: {trips_df.count()}")
# trips_df.printSchema()